<a href="https://colab.research.google.com/github/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/blob/main/Lecture-19-April-7-2026/Lecture-19_NeuralNetwork-OnMeltingPointDataset-ImprovedVersion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 19 - Neural Network Modelling on Melting Point Dataset - Improved Version

Here we are going to fit a deep neural network on the Bradley Melting Point Dataset, which is curated chemical dataset with melting points of around 3,000 chemical compounds, see [here](https://www.kaggle.com/datasets/aliffaagnur/melting-point-chemical-dataset/data).




Install RDKit and [LightGBM](https://lightgbm.readthedocs.io/en/stable/)

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install rdkit lightgbm mols2grid

Import all basic pacakges

In [ ]:
# basic
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# RDkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

# For progress bar
from tqdm.auto import tqdm

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor

# LightGBM
from lightgbm import LGBMRegressor, plot_importance


tqdm.pandas()

Download dataset

In [ ]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/refs/heads/main/Assignment-2/"
dataset_filename="BradleyDoublePlusGoodMeltingPointDataset.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

Read dataset

In [ ]:
data_mp = pd.read_csv("BradleyDoublePlusGoodMeltingPointDataset.csv")



In [ ]:
data_mp

In [ ]:
# Simplify by removing columns from the data frame
data_mp = data_mp.drop(columns=['csid','link','source','count','min','max','range'])

In [ ]:
property_names = list(rdMolDescriptors.Properties.GetAvailableProperties())
property_getter = rdMolDescriptors.Properties(property_names)

In [ ]:
def smi2props(smi):
    mol = Chem.MolFromSmiles(smi)
    props = None
    if mol:
        props = np.array(property_getter.ComputeProperties(mol))
    return props

In [ ]:
# one to the calculations for all smiles strings
data_mp['props'] = data_mp.smiles.progress_apply(smi2props)
data_mp = data_mp.dropna(subset=['props'])
data_mp[property_names] = data_mp['props'].tolist()
data_mp.drop("props",axis=1,inplace=True)

### Visualize Features and Target

## Train using LightGBM

In [ ]:
train, test = train_test_split(data_mp,test_size=0.20)

In [ ]:
train_X = train[property_names]
train_y = train.mpC
test_X = test[property_names]
test_y = test.mpC

In [ ]:
lgbm = LGBMRegressor()
lgbm.fit(train_X, train_y)

In [ ]:
test_pred = lgbm.predict(test_X)
train_pred = lgbm.predict(train_X)

In [ ]:
plt.plot(train_y, train_pred,'.',label='Train set')
plt.plot(test_y, test_pred,'.',label='Test set')
# to get a diagonal line
diagonal_line =np.linspace(np.min(train_y),np.max(train_y),1000)
plt.plot(diagonal_line,diagonal_line, color='red', linestyle='--')
# plt.axline([0, 0], slope=1, color='red', linestyle='--')
plt.legend()
plt.ylabel("Predicted Value")
plt.xlabel("Actual Value")
plt.show()

In [ ]:
print(train_X.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Check if CUDA is available
print(torch.cuda.is_available())  # True = GPU available

# See which device you're on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # 'cuda' or 'cpu'

# Get GPU name (if available)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # e.g. 'NVIDIA GeForce RTX 3090'
    print(torch.cuda.device_count())      # Number of GPUs

In [ ]:
class DeepNet(nn.Module):
    def __init__(self, input_dim: int,
                 output_dim: int = 1,
                 hidden_dim: int = 128,
                 n_layers: int = 6):
        super().__init__()
        layers = [nn.Linear(input_dim, hidden_dim), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

        # Xavier init for faster convergence
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

In [ ]:
train_X_t = torch.tensor(train_X.values, dtype=torch.float32)
train_y_t = torch.tensor(train_y.values, dtype=torch.float32).unsqueeze(1)
test_X_t = torch.tensor(test_X.values, dtype=torch.float32)
test_y_t = torch.tensor(test_y.values, dtype=torch.float32).unsqueeze(1)

# put the tensors onto the GPU for faster training
train_X_t = train_X_t.to(device)
train_y_t = train_y_t.to(device)
test_X_t = test_X_t.to(device)
test_y_t = test_y_t.to(device)



In [ ]:
print(test_y_t.shape)

In [ ]:
def regularization_loss(model: nn.Module, reg_type: str | None) -> torch.Tensor:
    """Return the regularization penalty for all weight tensors (biases excluded)."""
    if reg_type is None:
        return torch.tensor(0.0)

    weights = [p for name, p in model.named_parameters() if "weight" in name]

    if reg_type == "L1":
        return LAMBDA_L1 * sum(p.abs().sum() for p in weights)

    if reg_type == "L2":
        return LAMBDA_L2 * sum(p.pow(2).sum() for p in weights)

    if reg_type == "ElasticNet":
        l1 = LAMBDA_L1 * sum(p.abs().sum() for p in weights)
        l2 = LAMBDA_L2 * sum(p.pow(2).sum() for p in weights)
        return l1 + l2

    raise ValueError(f"Unknown reg_type '{reg_type}'. Choose None, 'L1', 'L2', or 'ElasticNet'.")

In [ ]:
def do_training(EPOCHS=10_000, LR=1e-3 , REG_TYPE=None):

  input_dim = train_X.shape[1]
  print(input_dim)

  model = DeepNet(input_dim=input_dim, hidden_dim=128, n_layers=3)

  # put the model onto the GPU for faster training
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)

  print(model)
  total_params = sum(p.numel() for p in model.parameters())
  print(f"\nTotal parameters: {total_params:,}\n")

  EPOCHS    = EPOCHS
  LR        = LR
  SCHEDULER_PATIENCE = 300

  # ── Regularization config ─────────────────────────────────────────────────────
  # Set REG_TYPE to: None | "L1" | "L2" | "ElasticNet"
  REG_TYPE   = REG_TYPE      # None | "L1" | "L2" | "ElasticNet"
  LAMBDA_L1  = 1e-4       # L1 penalty weight  (used for L1 and ElasticNet)
  LAMBDA_L2  = 1e-4       # L2 penalty weight  (used for L2 and ElasticNet)


  reg_label = REG_TYPE if REG_TYPE else "None"
  print(f"Regularization : {reg_label}")
  if REG_TYPE in ("L1", "ElasticNet"):
      print(f"  λ_L1 = {LAMBDA_L1}")
  if REG_TYPE in ("L2", "ElasticNet"):
      print(f"  λ_L2 = {LAMBDA_L2}")
  print()

  criterion = nn.MSELoss()
  optimizer = optim.Adam(model.parameters(), lr=LR)
  scheduler = optim.lr_scheduler.ReduceLROnPlateau(
      optimizer, patience=SCHEDULER_PATIENCE, factor=0.5
  )

  loss_history      = []   # total loss  (MSE + penalty)
  mse_history       = []   # data term only
  reg_loss_history  = []   # penalty term only

  loss_test_history      = []   # total loss  (MSE + penalty)
  mse_test_history       = []   # data term only
  reg_test_loss_history  = []   # penalty term only

  for epoch in range(1, EPOCHS + 1):
      # set the NN to training mode
      model.train()
      # set gradient to zero
      optimizer.zero_grad()

      # calculate the loss
      pred     = model(train_X_t)
      mse_loss = criterion(pred, train_y_t)
      reg_loss = regularization_loss(model, REG_TYPE)
      loss     = mse_loss + reg_loss

      # calculate the loss of the testing data
      pred_test     = model(test_X_t)
      mse_loss_test = criterion(pred_test, test_y_t)
      reg_loss_test = regularization_loss(model, REG_TYPE)
      loss_test     = mse_loss_test + reg_loss_test

      # calculate the gradient of the loss
      loss.backward()
      # update the parameters based on the loss
      optimizer.step()
      scheduler.step(loss)

      #
      loss_history.append(loss.item())
      mse_history.append(mse_loss.item())
      reg_loss_history.append(reg_loss.item())

      loss_test_history.append(loss_test.item())
      mse_test_history.append(mse_loss_test.item())
      reg_test_loss_history.append(reg_loss_test.item())

      if epoch % 500 == 0:
          print(f"Epoch {epoch:5d}/{EPOCHS} | Total: {loss.item():.6f} | "
                f"MSE: {mse_loss.item():.6f} | Reg ({reg_label}): {reg_loss.item():.6f} | "
                f"LR: {optimizer.param_groups[0]['lr']:.2e}")
  return model, loss_history, loss_test_history, mse_history, mse_test_history




In [ ]:
def make_plots(model, loss_history, loss_test_history, mse_history=None, mse_test_history=None):
  plt.plot(loss_test_history, label='Total - Test set')
  plt.plot(loss_history, label='Total - Train set')


  if mse_history is not None:
    plt.plot(mse_test_history, label='MSE - Test set')
    plt.plot(mse_history, label='MSE - Train set')


  plt.yscale('log')
  plt.legend()
  plt.ylabel("Loss")
  plt.xlabel("Epoch")
  plt.show()

  model.eval()
  with torch.no_grad():
      pred_test_y_t   = model(test_X_t)
      pred_train_y_t   = model(train_X_t)

  # need to use .cpu() on the tensors to move them from GPU to CPU memory
  plt.plot(pred_test_y_t.cpu(),test_y_t.cpu(),  '.',label='Test set')
  plt.plot(pred_train_y_t.cpu(),train_y_t.cpu(),'.',label='Train set')
  # to get a diagonal line
  diagonal_line =np.linspace(np.min(train_y),np.max(train_y),1000)
  plt.plot(diagonal_line,diagonal_line, color='red', linestyle='--')
  # plt.axline([0, 0], slope=1, color='red', linestyle='--')
  plt.legend()
  plt.xlabel("Predicted Value")
  plt.ylabel("Actual Value")
  plt.show()

  plt.plot(pred_test_y_t.cpu(),test_y_t.cpu()-pred_test_y_t.cpu(),  '.',label='Test set')
  plt.plot(pred_train_y_t.cpu(),train_y_t.cpu()-pred_train_y_t.cpu(),'.',label='Train set')
  plt.axhline(0, color='black', linewidth=1)
  plt.legend()
  plt.xlabel("Predicted Value")
  plt.ylabel("Residual (Actual-Predicted)")
  plt.show()

  sns.kdeplot(test_y_t.cpu()-pred_test_y_t.cpu(),label='Test set')
  plt.axvline(0, color='black', linewidth=1)
  plt.legend()
  plt.show()

  sns.kdeplot(train_y_t.cpu()-pred_train_y_t.cpu(),label='Train set')
  plt.axvline(0, color='black', linewidth=1)
  plt.legend()
  plt.show()



## Training without any regularization

In [ ]:
model, loss_history, loss_test_history, mse_history, mse_test_history = do_training(EPOCHS=50000)

In [ ]:
make_plots(model, loss_history, loss_test_history, mse_history=None, mse_test_history=None)

## Training with L2 regularization

In [ ]:
model, loss_history, loss_test_history, mse_history, mse_test_history = do_training(EPOCHS=20000, REG_TYPE='L2')

In [ ]:
make_plots(model, loss_history, loss_test_history, mse_history=mse_history, mse_test_history=mse_test_history)